In [1]:
import numpy as np
datapath = "../../Preprocessing/Encoded_Data/mopb_onehot_encoded.npz"
data = np.load(datapath)
embeddings = data["embeddings"]
labels = data["labels"]

In [7]:
counts = {}
for label in labels:
    counts[label] = counts[label] + 1 if label in counts else 1
print(counts)
print(len(counts))

{np.str_('DmsA'): 100, np.str_('NarG'): 100, np.str_('ActB'): 100, np.str_('Nqo3'): 100, np.str_('Fdhs'): 100, np.str_('NasCNarB'): 100, np.str_('FdhG'): 100, np.str_('FwdB'): 100, np.str_('TtrASrdA'): 100, np.str_('PsrAPhsA'): 100, np.str_('DorATorA'): 100, np.str_('NapA'): 100, np.str_('AH'): 100}
13


In [8]:
from sklearn.cluster import HDBSCAN
from sklearn.metrics import v_measure_score

hdb = HDBSCAN(
    min_cluster_size=10,
    min_samples=1,
    metric="euclidean",
    copy=True
)
hdb_labels = hdb.fit_predict(embeddings)
score = v_measure_score(labels, hdb_labels)

print("V-core:", score)
print("Number of clusters:", len(set(hdb_labels)))
print("Number of noise points:", len(hdb_labels == -1))

V-core: 0.24107658858356634
Number of clusters: 4
Number of noise points: 1300


In [9]:
from sklearn.cluster import HDBSCAN
from sklearn.metrics import v_measure_score

best_score = 0
best_params = {}
best_results = {}

for min_cluser_size in [5, 10, 20, 30, 50]:
    for min_samples in [1, 5, 10]:
        for metric in ["euclidean", "manhattan", "cosine"]:
            clusterer = HDBSCAN(
                min_cluster_size=min_cluser_size,
                min_samples=min_samples,
                metric=metric,
                copy=True
            )
            hdb_labels = clusterer.fit_predict(embeddings)

            if len(set(hdb_labels)) == 1: # only noise
                continue

            score = v_measure_score(labels, hdb_labels)

            if score > best_score:
                best_score = score
                best_params = {
                    'min_cluster_size': min_cluser_size,
                    'min_samples': min_samples,
                    'metric': metric,
                }
                best_results = {
                    'Number of cluster': len(set(hdb_labels)),
                    'Number of noise points': sum(hdb_labels == -1)
                }

print("Best score:", best_score)
print("Best parameters:", best_params)
print("Best result:", best_results)

Best score: 0.7123468678353582
Best parameters: {'min_cluster_size': 20, 'min_samples': 1, 'metric': 'cosine'}
Best result: {'Number of cluster': 12, 'Number of noise points': np.int64(156)}


### Soft predict for no -1 label

In [13]:
import hdbscan
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,
    min_samples=1,
    metric='cosine',
    algorithm='generic',
    prediction_data=True
)

predictions = clusterer.fit_predict(embeddings)
v_measure_score(labels, predictions)

/Users/teeratc/Desktop/2026 Spring/CS690U/CS690U-DGEB-project/.venv/lib/python3.11/site-packages/hdbscan/hdbscan_.py:1329: UserWarning: Metric cosine not supported for prediction data!
  warn("Metric {} not supported for prediction data!".format(self.metric))


0.7123468678353582

In [3]:
import hdbscan

best_score = 0
best_params = {}
best_results = {}

for min_cluster_size in [5, 10, 20, 30, 50]:
    for min_samples in [1, 5, 10]:
        for metric in ["euclidean", "manhattan"]:
            clusterer = hdbscan.HDBSCAN(
                    min_cluster_size=min_cluster_size,
                    min_samples=min_samples,
                    metric=metric,
                    prediction_data=True
                )

            predictions = clusterer.fit_predict(embeddings)

            soft_clusters = hdbscan.all_points_membership_vectors(clusterer)
            if soft_clusters.ndim == 1 or soft_clusters.shape[1] <= 1:
                no_noise_predictions = predictions
            else:
                best_cluster = np.argmax(soft_clusters, axis=1)
                no_noise_predictions = np.where(predictions == -1, best_cluster, predictions)

            if len(set(no_noise_predictions)) == 1: # only noise
                continue

            score = v_measure_score(labels, no_noise_predictions)

            if score > best_score:
                best_score = score
                best_params = {
                    'min_cluster_size': min_cluster_size,
                    'min_samples': min_samples,
                    'metric': metric,
                }
                best_results = {
                    'Number of cluster': len(set(no_noise_predictions)),
                    'Number of noise points': sum(no_noise_predictions == -1)
                }

print("Best score:", best_score)
print("Best parameters:", best_params)
print("Best result:", best_results)

Best score: 0.07699840886964483
Best parameters: {'min_cluster_size': 20, 'min_samples': 1, 'metric': 'euclidean'}
Best result: {'Number of cluster': 2, 'Number of noise points': np.int64(0)}
